In [20]:
import pandas as pd

# Read your CSV file directly
df = pd.read_csv("../tabular_3/diabetes_3.csv")

print(" File loaded successfully!")
print(f"Dataset shape: {df.shape}")


 File loaded successfully!
Dataset shape: (768, 9)


In [21]:
# Show all column names
print("Column names:")
print(df.columns.tolist())

# check data types
print("\nDtypes:")
print(df.dtypes)

# Preview first 5 rows
df.head()

Column names:
['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

Dtypes:
Pregnancies                   int64
Glucose                       int64
BloodPressure                 int64
SkinThickness                 int64
Insulin                       int64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                           int64
Outcome                       int64
dtype: object


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [22]:
import numpy as np
from pathlib import Path

# Target feature template (8 standardized features) ---
FEATURES = ['Age','Sex','BMI','GenHlth','HighBP','DiffWalk','HighChol','HeartDiseaseorAttack']

# Build aligned DataFrame (without mask columns) ---
aligned = pd.DataFrame(index=df.index, columns=FEATURES, dtype='float64')

# Directly mapped features
if 'Age' in df.columns: 
    aligned['Age'] = df['Age']
if 'BMI' in df.columns: 
    aligned['BMI'] = df['BMI']

# Sex: The Pima dataset contains only female participants → set Sex = 0
aligned['Sex'] = 0

# Derive HighBP from BloodPressure (>=90 → 1, else 0)
if 'BloodPressure' in df.columns:
    aligned['HighBP'] = (pd.to_numeric(df['BloodPressure'], errors='coerce') >= 90).astype('int8')

# Fill all remaining features with 0
aligned = aligned.fillna(0)

# Add label column from 'Outcome'
aligned['label'] = df['Outcome'].astype('int8')

print("Alignment completed successfully.")
print("Columns:", aligned.columns.tolist())
print("\nThis aligned dataset contains 8 standardized features without masks, ready for model training.")
aligned.head()


Alignment completed successfully.
Columns: ['Age', 'Sex', 'BMI', 'GenHlth', 'HighBP', 'DiffWalk', 'HighChol', 'HeartDiseaseorAttack', 'label']

This aligned dataset contains 8 standardized features without masks, ready for model training.


,Age,Sex,BMI,GenHlth,HighBP,DiffWalk,HighChol,HeartDiseaseorAttack,label
0,50,0,33.6,0.0,0,0.0,0.0,0.0,1
1,31,0,26.6,0.0,0,0.0,0.0,0.0,0
2,32,0,23.3,0.0,0,0.0,0.0,0.0,1
3,21,0,28.1,0.0,0,0.0,0.0,0.0,0
4,33,0,43.1,0.0,0,0.0,0.0,0.0,1


In [24]:
from pathlib import Path

# Save directly into existing folder "Datasets/tabular_3"
out_dir = Path("../tabular_3")

# Save aligned clean dataset (without masks)
master_path = out_dir / "tabular3_aligned_clean.csv"
aligned.to_csv(master_path, index=False)

print(f"Saved clean aligned dataset to: {master_path}")
print("Shape:", aligned.shape)


Saved clean aligned dataset to: ..\tabular_3\tabular3_aligned_clean.csv
Shape: (768, 9)


In [25]:
from sklearn.model_selection import train_test_split
from pathlib import Path
import pandas as pd

# Load the aligned clean dataset
df = pd.read_csv("../tabular_3/tabular3_aligned_clean.csv")

# Define split ratios (train, val, test)
split_ratio = (0.6, 0.2, 0.2)

# Split train vs temp (val + test)
train_size = split_ratio[0]
df_train, df_temp = train_test_split(
    df, test_size=1-train_size, random_state=42, stratify=df["label"]
)

# Split temp into val and test
val_ratio = split_ratio[1] / (split_ratio[1] + split_ratio[2])
df_val, df_test = train_test_split(
    df_temp, test_size=1-val_ratio, random_state=42, stratify=df_temp["label"]
)

# Save all splits into the same folder
out_dir = Path("../tabular_3")
train_path = out_dir / "diabetes_3_train.csv"
val_path = out_dir / "diabetes_3_val.csv"
test_path = out_dir / "diabetes_3_test.csv"

df_train.to_csv(train_path, index=False)
df_val.to_csv(val_path, index=False)
df_test.to_csv(test_path, index=False)

print("Saved train/val/test splits:")
print(f" - Train: {train_path}  shape={df_train.shape}")
print(f" - Val:   {val_path}  shape={df_val.shape}")
print(f" - Test:  {test_path}  shape={df_test.shape}")

# Optional: check label distribution balance
def label_ratio(df):
    return df["label"].value_counts(normalize=True).round(3).to_dict()

print("\nLabel distribution:")
print("Train:", label_ratio(df_train))
print("Val:  ", label_ratio(df_val))
print("Test: ", label_ratio(df_test))


Saved train/val/test splits:
 - Train: ..\tabular_3\diabetes_3_train.csv  shape=(460, 9)
 - Val:   ..\tabular_3\diabetes_3_val.csv  shape=(154, 9)
 - Test:  ..\tabular_3\diabetes_3_test.csv  shape=(154, 9)

Label distribution:
Train: {0: 0.65, 1: 0.35}
Val:   {0: 0.649, 1: 0.351}
Test:  {0: 0.656, 1: 0.344}
